# Relatório de Governança de Tags

**Objetivo:** consolidar e reportar todas as tags aplicadas nos objetos do Unity Catalog (catalog, schemas, tabelas), permitindo mensurar governança de dados de forma auditável — incluindo a aplicação de uma nova tag `camada`, identificando explicitamente a qual etapa da arquitetura Medallion (raw/bronze/silver/gold) cada schema pertence.

**Fonte:** `poc_latam_food.information_schema` (catalog_tags, schema_tags, table_tags).

**Limitação conhecida:** as tags aplicadas ao Job de orquestração (`ambiente`, `projeto`, `tipo`, `centro_custo`) não são consultáveis via este information_schema, pois pertencem à camada de Jobs/Workflows, não ao Unity Catalog. São documentadas manualmente nesta seção como referência complementar.

In [0]:
# Aplicação da tag 'camada' em cada schema, identificando a etapa Medallion correspondente

schemas_camadas = {
    "landing": "landing",
    "raw": "raw",
    "bronze": "bronze",
    "silver": "silver",
    "gold": "gold",
}

for schema, camada in schemas_camadas.items():
    spark.sql(f"ALTER SCHEMA poc_latam_food.{schema} SET TAGS ('camada' = '{camada}')")
    print(f"Tag 'camada={camada}' aplicada em poc_latam_food.{schema}")

In [0]:
# Consulta das tags de catalog e schema via information_schema

df_catalog_tags = spark.sql("""
    SELECT catalog_name, tag_name, tag_value
    FROM poc_latam_food.information_schema.catalog_tags
""")

df_schema_tags = spark.sql("""
    SELECT catalog_name, schema_name, tag_name, tag_value
    FROM poc_latam_food.information_schema.schema_tags
""")

print("=== Tags do Catalog ===")
df_catalog_tags.display()

print("=== Tags dos Schemas ===")
df_schema_tags.display()

In [0]:
# Consulta de tags em tabelas individuais + consolidação do relatório completo

df_table_tags = spark.sql("""
    SELECT catalog_name, schema_name, table_name, tag_name, tag_value
    FROM poc_latam_food.information_schema.table_tags
""")

print("=== Tags de Tabelas Individuais ===")
df_table_tags.display()

print(f"Total de tags em tabelas: {df_table_tags.count()}")

In [0]:
# Consolidação e gravação do relatório de governança

from pyspark.sql.functions import lit, col

df_catalog_relatorio = df_catalog_tags.select(
    lit("catalog").alias("nivel"),
    col("catalog_name"),
    lit(None).cast("string").alias("schema_name"),
    lit(None).cast("string").alias("table_name"),
    "tag_name",
    "tag_value"
)

df_schema_relatorio = df_schema_tags.select(
    lit("schema").alias("nivel"),
    "catalog_name",
    "schema_name",
    lit(None).cast("string").alias("table_name"),
    "tag_name",
    "tag_value"
)

df_table_relatorio = df_table_tags.select(
    lit("table").alias("nivel"),
    "catalog_name",
    "schema_name",
    "table_name",
    "tag_name",
    "tag_value"
)

df_relatorio_completo = df_catalog_relatorio.unionByName(df_schema_relatorio).unionByName(df_table_relatorio)

df_relatorio_completo.write.format("delta").mode("overwrite").saveAsTable("poc_latam_food.gold.governance_tags_report")

df_relatorio_completo.orderBy("nivel", "schema_name").display()

In [0]:
# Documentação complementar - tags do Job de orquestração (não disponíveis via information_schema)

tags_job_workflow = {
    "ambiente": "poc",
    "projeto": "latam-food-lakehouse",
    "tipo": "definitivo",
    "centro_custo": "pipeline-principal",
}

print("=== Tags do Job 'Pipeline Diario - POC LATAM Food' (documentado manualmente) ===")
for chave, valor in tags_job_workflow.items():
    print(f"{chave}: {valor}")

print("\nNota: estas tags pertencem à camada de Jobs/Workflows do Databricks, "
      "não ao Unity Catalog, e por isso não são consultáveis via information_schema. "
      "Documentadas aqui manualmente para completar a visão de governança do projeto.")